# HW2 Stern-Gerlach Application

PDF pages 3-47: basic matrix operations, Born rule, projectors, conditional states, order effects, unitary matrices, spectral decomposition, and the A-B-C Stern-Gerlach example.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import csv
import numpy as np
np.set_printoptions(precision=4, suppress=True)

from quantum_lib import (
    binary_observable, born_rule, conditional_state, dagger, event_probability,
    interference, is_unitary, kron, matmul, normalize, order_effects, outer_product,
    projector, ray_projector, sequential_probability, total_probability, transpose,
    two_dim_unitary, unitary_from_hamiltonian, unitary_from_spectral,
)


## Basic Matrix Operations


In [2]:
A = np.array([[1, 2j], [-1j, 3]], dtype=complex)
B = np.array([[0, 1], [1, 0]], dtype=complex)
ket0 = np.array([1, 0], dtype=complex)
ket1 = np.array([0, 1], dtype=complex)

print('transpose(A)=\n', transpose(A))
print('dagger(A)=\n', dagger(A))
print('A @ B=\n', matmul(A, B))
print('<0|1>=', np.vdot(ket0, ket1))
print('|0><1|=\n', outer_product(ket0, ket1))
print('norm(A)=', np.linalg.norm(A))
print('eig(A)=', np.linalg.eig(A)[0])
print('trace(A)=', np.trace(A))
print('kron(|0>, |1>)=', kron(ket0, ket1))


transpose(A)=
 [[ 1.+0.j -0.-1.j]
 [ 0.+2.j  3.+0.j]]
dagger(A)=
 [[ 1.-0.j -0.+1.j]
 [ 0.-2.j  3.-0.j]]
A @ B=
 [[0.+2.j 1.+0.j]
 [3.+0.j 0.-1.j]]
<0|1>= 0j
|0><1|=
 [[0.+0.j 1.+0.j]
 [0.+0.j 0.+0.j]]
norm(A)= 3.872983346207417
eig(A)= [0.2679-0.j 3.7321+0.j]
trace(A)= (4+0j)
kron(|0>, |1>)= [0.+0.j 1.+0.j 0.+0.j 0.+0.j]


## Events, Projectors, Born Rule, and Conditional States


In [3]:
psi5 = normalize([np.sqrt(.10), np.sqrt(.20), np.sqrt(.25), np.sqrt(.30), np.sqrt(.15)])
P_A = projector([3, 4], 5)       # composite event, e.g. R > 3
P_not_A = np.eye(5) - P_A
P_D = projector([2, 3], 5)       # decision-like event in the rating basis

print('psi probabilities:', born_rule(psi5))
print('q(A)=', event_probability(psi5, P_A))
print('q(not A)=', event_probability(psi5, P_not_A))
print('conditional psi | A =', conditional_state(psi5, P_A))
print('q(A then D)=', sequential_probability(psi5, P_A, P_D))
print('q(D then A)=', sequential_probability(psi5, P_D, P_A))
print('qT(D)=', total_probability(psi5, [P_A, P_not_A], P_D))
print('q(D), qT(D), interference=', interference(psi5, [P_A, P_not_A], P_D))


psi probabilities: [0.1  0.2  0.25 0.3  0.15]
q(A)= 0.4500000000000002
q(not A)= 0.5500000000000003
conditional psi | A = [0.    +0.j 0.    +0.j 0.    +0.j 0.8165+0.j 0.5774+0.j]
q(A then D)= 0.3000000000000001
q(D then A)= 0.3000000000000001
qT(D)= 0.5500000000000003
q(D), qT(D), interference= (0.5500000000000003, 0.5500000000000003, 0.0)


## Noncommuting Decision Basis and Order Effect


In [5]:
U5 = np.eye(5, dtype=complex)
angle = np.deg2rad(35)
U5[np.ix_([1, 3], [1, 3])] = [[np.cos(angle), -np.sin(angle)], [np.sin(angle), np.cos(angle)]]
P_D_rotated = U5 @ P_D @ dagger(U5)

print('U unitary:', is_unitary(U5))
print('commutator norm ||PA PD - PD PA|| =', np.linalg.norm(P_A @ P_D_rotated - P_D_rotated @ P_A))
print('q(A,D), q(D,A), difference =', order_effects(psi5, P_A, P_D_rotated))


U unitary: True
commutator norm ||PA PD - PD PA|| = 0.6644630243886747
q(A,D), q(D,A), difference = (0.20130302149885038, 0.02477655274277444, 0.17652646875607594)


## Hamiltonian, Unitary Matrix, and Spectral Decomposition


In [6]:
H = np.array([[1, 1j], [-1j, 2]], dtype=complex)
U_expm = unitary_from_hamiltonian(H, t=.4)
U_spec = unitary_from_spectral(H, t=.4)

print('H Hermitian:', np.allclose(H, dagger(H)))
print('U from expm unitary:', is_unitary(U_expm))
print('spectral and expm agree:', np.allclose(U_expm, U_spec))
print('eigenvalues:', np.linalg.eigh(H)[0])


H Hermitian: True
U from expm unitary: True
spectral and expm agree: True
eigenvalues: [0.382 2.618]


## Empirical A-B-C S-G Example


In [7]:
values = {}
with open(PROJECT_ROOT / 'data' / 'hw2_table_2_3.csv', newline='') as f:
    for row in csv.DictReader(f):
        values[row['quantity']] = float(row['value'])

p_A_plus = values['p_A_plus']
psi_A = np.array([np.sqrt(p_A_plus), np.sqrt(1 - p_A_plus)], dtype=complex)
PA_plus = projector([0], 2)
PA_minus = projector([1], 2)

theta_b = np.deg2rad(values['theta_b_degrees'])
phi_b = np.deg2rad(values['phi_b_degrees'])
U_B = two_dim_unitary(theta_b, phi_b)
PB_plus = ray_projector(U_B[:, 0])
PB_minus = ray_projector(U_B[:, 1])
B_hat = binary_observable(theta_b, phi_b)

theta_c = np.deg2rad(values['theta_c_degrees'])
phi_c = np.deg2rad(values['phi_c_degrees'])
U_C = two_dim_unitary(theta_c, phi_c)
PC_plus = ray_projector(U_C[:, 0])
PC_minus = ray_projector(U_C[:, 1])
C_hat = binary_observable(theta_c, phi_c)

print('initial psi_A:', psi_A)
print('qA(+), qA(-):', event_probability(psi_A, PA_plus), event_probability(psi_A, PA_minus))
print('U_B unitary:', is_unitary(U_B), 'U_C unitary:', is_unitary(U_C))
print('B observable=\n', B_hat)
print('C observable=\n', C_hat)
print('q(A+,B+)=', sequential_probability(psi_A, PA_plus, PB_plus))
print('q(A+,B-,C+)=', sequential_probability(psi_A, PA_plus, PB_minus, PC_plus))
print('q(A+,B+,C+)=', sequential_probability(psi_A, PA_plus, PB_plus, PC_plus), '(target around 0.29 from slides)')


initial psi_A: [0.922 +0.j 0.3873+0.j]
qA(+), qA(-): 0.8500000000000001 0.15000000000000002
U_B unitary: True U_C unitary: True
B observable=
 [[ 0.6351+0.j  0.7724+0.j]
 [ 0.7724+0.j -0.6351+0.j]]
C observable=
 [[-0.1557-0.j     -0.0662-0.9856j]
 [-0.0662+0.9856j  0.1557-0.j    ]]
q(A+,B+)= 0.6949323871249284
q(A+,B-,C+)= 0.08916510931022878
q(A+,B+,C+)= 0.2953407437613812 (target around 0.29 from slides)
